# Lesson 10 - Agents IA en Production

Dans cette leçon, vous apprendrez les **modèles de production** pour les agents IA en utilisant le Microsoft Agent Framework (Python). Nous couvrons :

- **Observabilité** — ajout de mesures temporelles et de journalisation aux interactions des agents
- **Évaluation** — utilisation d’un agent évaluateur pour noter la qualité des réponses
- **Gestion des coûts** — stratégies pour l’optimisation des tokens et la sélection des modèles

Le scénario est un **agent de voyage** qui aide les utilisateurs à planifier des voyages, avec une supervision et une évaluation intégrées.


## Configuration


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Considérations pour la production

Passer des agents IA des prototypes à la production nécessite une attention particulière à trois piliers :

1. **Observabilité** — Vous avez besoin de visibilité sur ce que fait l'agent, combien de temps cela prend, et quels outils il appelle. Sans traçage et journalisation, déboguer les problèmes en production est presque impossible.

2. **Évaluation** — Les contrôles qualité automatisés garantissent que les réponses de l'agent restent précises, complètes et utiles au fil du temps. Un agent évaluateur peut attribuer des scores aux réponses selon des critères définis.

3. **Gestion des coûts** — L'utilisation des tokens impacte directement le coût. Des stratégies comme l'optimisation des requêtes, la sélection du modèle et la mise en cache aident à maîtriser les dépenses sans sacrifier la qualité.


## Construire un agent observable

Nous définissons des outils de voyage et encapsulons l'appel de l'agent avec un chronométrage afin que nous puissions surveiller la latence. En production, vous intégreriez OpenTelemetry ou un backend de traçage similaire.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Modèles d'évaluation

Un modèle courant en production est d'utiliser un second agent comme **évaluateur**. L'évaluateur note la réponse de l'agent principal selon des critères prédéfinis tels que l'exhaustivité, la précision et l'utilité.

Cela permet :
- Des contrôles qualité automatisés avant que les réponses n'atteignent les utilisateurs
- La détection de régressions lorsque les invites ou les modèles changent
- La surveillance continue des performances de l'agent au fil du temps


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Stratégies de gestion des coûts

Le contrôle des coûts est essentiel pour les agents d’IA de production. Voici les stratégies clés :

| Stratégie | Description |
|---|---|
| **Optimisation des prompts** | Gardez les instructions système concises. Supprimez le contexte redondant pour réduire les tokens d’entrée. |
| **Sélection du modèle** | Utilisez des modèles plus petits et moins chers (par exemple GPT-4o-mini) pour des tâches simples comme la classification ou l’extraction, et réservez les modèles plus grands pour le raisonnement complexe. |
| **Mise en cache** | Mettez en cache les résultats des outils et les requêtes fréquentes pour éviter les appels API redondants. |
| **Budgets de tokens** | Définissez des limites `max_tokens` pour éviter des réponses exceptionnellement longues. |
| **Regroupement** | Regroupez plusieurs requêtes utilisateur en un seul appel API quand c’est possible. |

En pratique, une approche en plusieurs niveaux fonctionne bien : dirigez les requêtes simples vers un modèle rapide et peu coûteux et ne faites monter en charge que les requêtes complexes vers un modèle plus performant.


## Résumé

Dans cette leçon, vous avez appris à :

1. **Ajouter de l'observabilité** aux interactions des agents avec le chronométrage et la journalisation, posant les bases du traçage et de la surveillance.
2. **Évaluer automatiquement les réponses des agents** à l'aide d'un agent évaluateur qui attribue une note à la complétude, la précision et l'utilité.
3. **Gérer les coûts** grâce à l'optimisation des invites, la sélection des modèles, la mise en cache et les budgets de jetons.

Ces modèles de production aident à garantir que vos agents IA sont fiables, mesurables et rentables à grande échelle.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
